# Premotor PCA by neuron type
Choose **Motor, Sensory, Sensorimotor, or None**, retain sessions with at least
**five neurons of that type**, and fit separate tracking/playback PCAs. Start
position, final position and movement direction each have an interactive 3D
viewer and their own descriptive regression or logistic-regression analysis.

Run top to bottom with the project's `.venv` kernel. Edit the settings cell to
change neuron type, session cutoff or baseline subtraction, then rerun the cells
below it. The default is **Sensorimotor, minimum 5, baseline-subtracted**.

This notebook reuses the saved raw premotor extraction. It never changes source
bout tables, spike counts, classifications, or the other notebooks' results.
Models, selected neuron IDs, settings, and evaluations are saved in a separate
run directory identified by type, cutoff, analysis window and preprocessing.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from importlib import reload
import sklearn

cwd = Path.cwd().resolve()
repo_dir = next(
    (candidate for parent in (cwd, *cwd.parents)
     for candidate in (parent, parent / "etheremin_project")
     if (candidate / "movement_bouts" / "utils.py").is_file()), None,
)
if repo_dir is None:
    raise FileNotFoundError("Run from the eTheremin workspace or inside the repository.")
sys.path.insert(0, str(repo_dir / "movement_bouts"))
import utils
import evaluation
reload(utils)
reload(evaluation)
from utils import prepare_pca_input, fit_pca_by_condition, make_interactive_pca_browser

data_dir = repo_dir.parent / "data"
bout_dir = data_dir / "derived" / "movement_bouts"
premotor_dir = data_dir / "derived" / "premotor_activity"

## 1. Settings
`baseline_subtracted` subtracts each bout/neuron's own mean rate over **[-1, -0.7) s**
from its analysis-window rates. `raw` uses uncorrected firing rates. This is
subtraction in spikes/s, not a ratio or z-score. PCA itself always centers each
condition, without variance standardization or whitening. No Gaussian smoothing
is used for this analysis: counts are averaged into five 100 ms windows.

The session cutoff is applied to the number of neurons with a matching type,
not to the total recorded unit count. Missing classifications are excluded;
**None** means the assigned cluster 3, never an absent classification.

In [ ]:
NEURON_TYPE = "Sensorimotor"  # "Motor", "Sensory", "Sensorimotor", "None"
MIN_NEURONS = 5              # Minimum selected neurons per session (>=3 for 3D).
ACTIVITY_MODE = "raw"  # "raw" or "baseline_subtracted"
ANALYSIS_WINDOW_S = (-0.5, 0.0)
BASELINE_WINDOW_S = (-1.0, -0.7)
N_BINS = 5                # 5 = 100 ms, 2 = 250 ms, 1 = 500 ms over [-0.5, 0).
EXTRACTION_SUBDIR = "extracted_-1.2_0"
SAVE_RESULTS = True

# Pooled evaluations use PC1-PC3 by default, independently per session/condition.
EVALUATION_COLUMNS = ["PC1", "PC2", "PC3"]
DESTINATION_MIN_DISPLACEMENT = None  # 5.0 reproduces the earlier abs(end-start)>5 analysis.
DIRECTION_C = 1.0                   # Inverse L2 penalty strength.
LINEAR_SCORE = "r2_adjusted"        # Or "r2".

In [ ]:
extraction_dir = premotor_dir / EXTRACTION_SUBDIR
extraction_parameters = json.loads((extraction_dir / "parameters.json").read_text(encoding="utf-8"))
# Bout IDs are meaningful only for the exact source tables used for extraction.
for filename, expected_hash in extraction_parameters["source_sha256"].items():
    actual_hash = hashlib.sha256((bout_dir / filename).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f"{filename} has changed since extraction; regenerate aligned counts first.")
movement_bouts = pd.read_pickle(bout_dir / "movement_bouts.pkl")
session_info = pd.read_pickle(bout_dir / "session_info.pkl")
neuron_ids_by_session = pd.read_pickle(extraction_dir / "neuron_ids_by_session.pkl")
classification_path = data_dir / "df_clus_encoding_phd.pkl"
classification_table = pd.read_pickle(classification_path)
DT = extraction_parameters["DT"]
print(f"{len(movement_bouts):,} reference bouts; {len(session_info)} sessions; DT={DT:g} s.")

## 2. Match neuron IDs and select sessions
The table below records the type counts and missing classifications for every
session, preserving the original session index `idx`. `selected_neuron_ids`
keeps the matching IDs in their original recording-axis order.

In [ ]:
category_to_cluster = {"Motor": 0, "Sensory": 1, "Sensorimotor": 2, "None": 3}
if NEURON_TYPE not in category_to_cluster:
    raise ValueError(f"Choose a neuron type from {list(category_to_cluster)}.")
if not isinstance(MIN_NEURONS, (int, np.integer)) or isinstance(MIN_NEURONS, bool) or MIN_NEURONS < 3:
    raise ValueError("MIN_NEURONS must be an integer >=3 for the 3D viewers.")
if ACTIVITY_MODE not in ("raw", "baseline_subtracted"):
    raise ValueError("ACTIVITY_MODE must be raw or baseline_subtracted.")
if classification_table["neuron_idx"].isna().any() or classification_table["neuron_idx"].duplicated().any():
    raise ValueError("Classification neuron_idx values must be unique and nonmissing.")
if not classification_table["cluster_encoding"].isin(category_to_cluster.values()).all():
    raise ValueError("Unknown or missing cluster_encoding in the classification table.")
cluster_by_neuron = classification_table.set_index("neuron_idx")["cluster_encoding"]
selected_neuron_ids = {}
selection_rows = []
for idx, info in session_info.iterrows():
    neuron_ids = pd.Index(neuron_ids_by_session.loc[idx, "neuron_ids"], name="neuron_id")
    if not neuron_ids.is_unique:
        raise ValueError(f"Duplicate recorded neuron ID in session {idx}.")
    clusters = pd.Series(neuron_ids.map(cluster_by_neuron), index=neuron_ids)
    selected = neuron_ids[clusters.eq(category_to_cluster[NEURON_TYPE]).to_numpy()].tolist()
    selected_neuron_ids[idx] = selected
    selection_rows.append({
        "idx": idx, "n_total": len(neuron_ids), "n_unclassified": int(clusters.isna().sum()),
        **{f"n_{category}": int(clusters.eq(cluster).sum())
           for category, cluster in category_to_cluster.items()},
        "n_selected": len(selected), "eligible": len(selected) >= MIN_NEURONS,
    })
session_selection = session_info.join(pd.DataFrame(selection_rows).set_index("idx"), validate="one_to_one")
eligible_sessions = session_selection.loc[session_selection["eligible"]].copy()
coverage_by_type = pd.DataFrame({
    "n_units": {category: int(session_selection[f"n_{category}"].sum()) for category in category_to_cluster},
    "eligible_sessions": {category: int(session_selection[f"n_{category}"].ge(MIN_NEURONS).sum())
                          for category in category_to_cluster},
})
display(coverage_by_type)
print(f"{NEURON_TYPE}: {len(eligible_sessions)} / {len(session_info)} sessions have >= {MIN_NEURONS} matching neurons.")
display(session_selection[["animal", "headstage", "session", "n_total", "n_selected", "n_unclassified", "eligible"]])
if eligible_sessions.empty:
    raise ValueError("No sessions meet the selected type/count cutoff; change the settings and rerun.")

## 3. Fit and save the selected-neuron PCAs
Prepare rates with the same helper as the original analysis, then select **only
the chosen neuron columns before fitting**. Each observation remains a bout/time
window, indexed by `(bout_id, bin_start_s)`. Tracking and playback have separate
PCA bases; neurons and coordinates are never concatenated across sessions.

The saved extraction is loaded once (approximately 0.9 GB). Averaging and PCA use
one session at a time. Bouts and sessions with no usable extraction are reported.

In [ ]:
# Snapshot the effective analysis settings and IDs in the fitted bundle.
run_settings = {
    "neuron_type": NEURON_TYPE, "cluster_encoding": category_to_cluster[NEURON_TYPE],
    "min_neurons": MIN_NEURONS, "DT": DT, "N_BINS": N_BINS,
    "activity_mode": ACTIVITY_MODE, "analysis_window_s": list(ANALYSIS_WINDOW_S),
    "baseline_window_s": list(BASELINE_WINDOW_S) if ACTIVITY_MODE == "baseline_subtracted" else None,
    "extraction_directory": str(extraction_dir), "extraction_parameters": extraction_parameters,
    "classification_file": str(classification_path),
    "classification_sha256": hashlib.sha256(classification_path.read_bytes()).hexdigest(),
    "category_to_cluster": category_to_cluster.copy(),
    "standardized": False, "whiten": False, "sklearn_version": sklearn.__version__,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
neuronal_bouts = pd.read_pickle(extraction_dir / "neuronal_bouts.pkl")
pca_sessions = {}
skipped_sessions = {
    idx: f"Only {row.n_selected} {NEURON_TYPE} neurons; minimum {MIN_NEURONS}"
    for idx, row in session_selection.loc[~session_selection["eligible"]].iterrows()
}
for idx in eligible_sessions.index:
    bout_ids = movement_bouts.index[movement_bouts["idx"].eq(idx)]
    if not bout_ids.isin(neuronal_bouts.index).any():
        skipped_sessions[idx] = "No complete extracted bouts"
        continue
    session_input, session_labels = prepare_pca_input(
        neuronal_bouts, movement_bouts, neuron_ids_by_session,
        idx=idx, dt=run_settings["DT"], n_bins=run_settings["N_BINS"],
        extraction_window_s=extraction_parameters["window_s"],
        analysis_window_s=run_settings["analysis_window_s"],
        baseline_window_s=run_settings["baseline_window_s"], activity_mode=run_settings["activity_mode"],
    )
    session_input = session_input.loc[:, selected_neuron_ids[idx]]
    fits = fit_pca_by_condition(session_input, session_labels)
    if fits:
        pca_sessions[idx] = fits
    else:
        skipped_sessions[idx] = "Fewer than two observations in either condition"
if not pca_sessions:
    raise ValueError("No PCA could be fitted; inspect skipped_sessions and the extraction.")
pca_cache = {
    "settings": run_settings, "sessions": pca_sessions,
    "session_info": session_selection.loc[list(pca_sessions)].copy(),
    "session_selection": session_selection.copy(), "skipped_sessions": skipped_sessions,
    "selected_neuron_ids": {idx: selected_neuron_ids[idx].copy() for idx in pca_sessions},
    "position_limits": (float(movement_bouts["initial_position"].min()),
                        float(movement_bouts["initial_position"].max())),
}
# Keep fit-level diagnostics explicit, including conditions absent from a session.
pca_rows = []
for idx in eligible_sessions.index:
    for condition in (0, 1):
        fit = pca_sessions.get(idx, {}).get(condition)
        pca_rows.append({
            "idx": idx, "condition": condition, "n_neurons": len(selected_neuron_ids[idx]),
            "n_observations": len(fit["results"]) if fit else 0,
            "n_bouts": fit["results"].index.get_level_values("bout_id").nunique() if fit else 0,
            "n_components": fit["model"].n_components_ if fit else 0,
            "variance_PC1_PC3": float(fit["model"].explained_variance_ratio_[:3].sum()) if fit else np.nan,
            "status": "ok" if fit else "skipped",
        })
pca_fit_summary = pd.DataFrame(pca_rows).set_index(["idx", "condition"])
pca_cache["fit_summary"] = pca_fit_summary
# The large raw extraction can be released once all models/results are fitted.
del neuronal_bouts
print(f"Fitted {sum(len(fits) for fits in pca_sessions.values())} PCAs across {len(pca_sessions)} sessions.")
display(pca_fit_summary)

In [ ]:
saved = pca_cache["settings"]
run_name = (
    f"neuron_type_{saved['neuron_type'].lower()}_min{saved['min_neurons']}_"
    f"analysis_{saved['analysis_window_s'][0]:g}_{saved['analysis_window_s'][1]:g}_"
    f"{saved['N_BINS']}bins_{saved['activity_mode']}"
)
if saved["baseline_window_s"] is not None:
    run_name += f"_{saved['baseline_window_s'][0]:g}_{saved['baseline_window_s'][1]:g}"
run_dir = premotor_dir / "runs" / run_name
pca_path = run_dir / "pca" / f"pca_{saved['N_BINS']}bins.pkl"
evaluation_dir = run_dir / "evaluation"
if SAVE_RESULTS:
    pca_path.parent.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(pca_cache, pca_path)
    print(f"Saved PCA models, labels, selected IDs, eligibility and settings to {pca_path}")
else:
    print("Results remain in memory; SAVE_RESULTS is False.")

## 4. Starting position: PCA and linear regression
Color reflects the saved position at movement onset. Each viewer has independent
session controls and linked cameras within its tracking/playback panels.
Regressions use the selected PCs and **all time windows pooled**, one ordinary
linear regression per session/condition. Starting-position regression has no
displacement filter. The summary and paired plots show adjusted R2 by default.

All scores here are **descriptive, in-sample**. There is no held-out validation,
randomization or significance test. Windows of a bout are dependent; row-based
adjusted R2 does not correct that dependence. Bout and observation counts are
reported separately. Left/right classification uses the same convention below.

In [ ]:
if "start_browser" in globals():
    for child in start_browser.children:
        child.close()
    start_browser.close()
start_browser = make_interactive_pca_browser(pca_cache, color_by="position", width=1400)
display(start_browser)

In [ ]:
start_regressions = evaluation.run_label_regressions(
    pca_cache, target="initial_position", columns=EVALUATION_COLUMNS,
    min_displacement=None,
)
start_regressions["settings"]["saved_at_utc"] = datetime.now(timezone.utc).isoformat()
if SAVE_RESULTS:
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    settings = start_regressions["settings"]
    cutoff = settings["min_displacement"]
    filter_tag = "all_bouts" if cutoff is None else f"absdelta_gt{cutoff:g}"
    start_regression_path = evaluation_dir / (
        f"regression_initial_position_{'-'.join(settings['columns'])}_{filter_tag}_pooled.pkl"
    )
    pd.to_pickle(start_regressions, start_regression_path)
print(start_regressions["summary"]["status"].value_counts())
display(start_regressions["summary"])

In [ ]:
from importlib import reload
import evaluation
reload(evaluation)  # Refresh animal labels/boundaries in an existing kernel.

fig_start_summary, _ = evaluation.plot_regression_summary(start_regressions["summary"], score=LINEAR_SCORE)
fig_start_summary.suptitle(
    f"Starting position | {pca_cache['settings']['neuron_type']} | {pca_cache['settings']['activity_mode']} | pooled windows"
)
fig_start_summary.tight_layout()
plt.show()
try:
    fig_start_paired, _, start_paired_scores = evaluation.plot_paired_regressions(
        start_regressions["summary"], score=LINEAR_SCORE,
    )
except ValueError as error:
    print(f"Paired plot unavailable: {error}")
else:
    fig_start_paired.suptitle("Starting position: matched tracking/playback scores")
    fig_start_paired.tight_layout()
    plt.show()
    print(f"Complete pairs: {len(start_paired_scores)}")
    display(start_paired_scores)

## 5. Final position: PCA and linear regression
Color reflects the observed movement endpoint, not an inferred target. Unknown
endpoints are gray and excluded from destination regression only. The PCA viewer
shows all observations; `DESTINATION_MIN_DISPLACEMENT` optionally restricts only
the regression to `abs(final_position - initial_position) > cutoff`. Default None
includes every bout with a finite endpoint. Setting 5 reproduces the earlier
filtered destination analysis. Neither this filter nor a missing endpoint refits PCA.

In [ ]:
if "end_browser" in globals():
    for child in end_browser.children:
        child.close()
    end_browser.close()
end_browser = make_interactive_pca_browser(pca_cache, color_by="destination", width=1400)
display(end_browser)

In [ ]:
end_regressions = evaluation.run_label_regressions(
    pca_cache, target="final_position", columns=EVALUATION_COLUMNS,
    min_displacement=DESTINATION_MIN_DISPLACEMENT,
)
end_regressions["settings"]["saved_at_utc"] = datetime.now(timezone.utc).isoformat()
if SAVE_RESULTS:
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    settings = end_regressions["settings"]
    cutoff = settings["min_displacement"]
    filter_tag = "all_bouts" if cutoff is None else f"absdelta_gt{cutoff:g}"
    end_regression_path = evaluation_dir / (
        f"regression_final_position_{'-'.join(settings['columns'])}_{filter_tag}_pooled.pkl"
    )
    pd.to_pickle(end_regressions, end_regression_path)
print(end_regressions["summary"]["status"].value_counts())
display(end_regressions["summary"])

In [ ]:
from importlib import reload
import evaluation
reload(evaluation)  # Refresh animal labels/boundaries in an existing kernel.

fig_end_summary, _ = evaluation.plot_regression_summary(end_regressions["summary"], score=LINEAR_SCORE)
fig_end_summary.suptitle(
    f"Final position | {pca_cache['settings']['neuron_type']} | {pca_cache['settings']['activity_mode']} | pooled windows"
)
fig_end_summary.tight_layout()
plt.show()
try:
    fig_end_paired, _, end_paired_scores = evaluation.plot_paired_regressions(
        end_regressions["summary"], score=LINEAR_SCORE,
    )
except ValueError as error:
    print(f"Paired plot unavailable: {error}")
else:
    fig_end_paired.suptitle("Final position: matched tracking/playback scores")
    fig_end_paired.tight_layout()
    plt.show()
    print(f"Complete pairs: {len(end_paired_scores)}")
    display(end_paired_scores)

## 6. Movement direction: PCA and logistic regression
Blue = left (-1), red = right (+1), gray = zero/unknown. This is **movement
direction**, not the side of starting position. Unknown directions are excluded
from the classifier; each fitted subset must contain both directions.

As before, use PC1-PC3 by default in a StandardScaler + L2 logistic pipeline,
with balanced class weights and `C=1`. PCA is not standardized or refitted.
Balanced accuracy and left/right recalls are in-sample descriptive scores.
A balanced-accuracy score of 0.5 is the constant-classifier reference. No
position or displacement filter is applied to the classifier.

In [ ]:
if "direction_browser" in globals():
    for child in direction_browser.children:
        child.close()
    direction_browser.close()
direction_browser = make_interactive_pca_browser(pca_cache, color_by="direction", width=1400)
display(direction_browser)

In [ ]:
direction_logistics = evaluation.run_direction_logistics(
    pca_cache, columns=EVALUATION_COLUMNS, C=DIRECTION_C,
)
direction_logistics["settings"]["saved_at_utc"] = datetime.now(timezone.utc).isoformat()
if SAVE_RESULTS:
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    settings = direction_logistics["settings"]
    direction_logistic_path = evaluation_dir / (
        f"logistic_initial_direction_{'-'.join(settings['columns'])}_C{settings['C']:g}_pooled.pkl"
    )
    pd.to_pickle(direction_logistics, direction_logistic_path)
print(direction_logistics["summary"]["status"].value_counts())
display(direction_logistics["summary"])

In [ ]:
from importlib import reload
import evaluation
reload(evaluation)  # Refresh animal labels/boundaries in an existing kernel.

fig_direction_summary, _ = evaluation.plot_regression_summary(direction_logistics["summary"], score="balanced_accuracy")
fig_direction_summary.suptitle(
    f"Left/right direction | {pca_cache['settings']['neuron_type']} | {pca_cache['settings']['activity_mode']} | pooled windows"
)
fig_direction_summary.tight_layout()
plt.show()
try:
    fig_direction_paired, _, direction_paired_scores = evaluation.plot_paired_regressions(
        direction_logistics["summary"], score="balanced_accuracy",
    )
except ValueError as error:
    print(f"Paired plot unavailable: {error}")
else:
    fig_direction_paired.suptitle("Left/right direction: matched tracking/playback scores")
    fig_direction_paired.tight_layout()
    plt.show()
    print(f"Complete pairs: {len(direction_paired_scores)}")
    display(direction_paired_scores)

## Compare neuron groups
The matched comparisons between neuron types and the full session are now in
[neuron_group_comparison.ipynb](neuron_group_comparison.ipynb), which runs independently.


## Reload an existing run
The saved PCA bundle contains models, coordinate/label tables, selected neuron
IDs and all settings. Reload it to use the three viewers without loading counts
or fitting again. Evaluation pickle files contain summaries, fitted models and
indexed predictions. Changing neuron type, cutoff or preprocessing requires new
PCA fits; recoloring the same results does not.

In [ ]:
# pca_cache = pd.read_pickle(pca_path)
# start_regressions = pd.read_pickle(start_regression_path)
# end_regressions = pd.read_pickle(end_regression_path)
# direction_logistics = pd.read_pickle(direction_logistic_path)